# Lesson 18: 使用密碼學收據保護 AI 代理

## 實作筆記本

本筆記本將帶您完成四個練習：

1. <strong>簽署您的第一個收據</strong> 用於代理工具調用並驗證它。
2. <strong>篡改收據</strong> 並觀察驗證失敗。
3. <strong>建立三收據鏈</strong> 並確認鏈的完整性。
4. **包裝 Microsoft Agent Framework 工具調用** 使每個動作皆發出收據。

所有密碼學原語均從良好維護的庫中導入（`pynacl` 用於 Ed25519，`jcs` 用於 RFC 8785 正規化 JSON，Python 標準庫的 `hashlib` 用於 SHA-256）。收據邏輯本身是純 Python，您可以閱讀和修改。

請依序執行每個單元。每個章節短且自成一體。


## 設定

安裝這兩個依賴。兩者皆擁有寬鬆的授權（Apache-2.0 / MIT）。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

這兩個輔助工具處理 base64url 編碼（無填充）和任意物件的 SHA-256 雜湊。他們讓筆記本的其餘部分可以專注於收據邏輯本身。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 第 1 節：簽署您的第一張收據

想像我們的 **Contoso Travel** 代理剛幫客戶查詢從悉尼飛往洛杉磯的航班。我們想將這次工具呼叫記錄為帶簽名的收據，讓未來審核員能在不必信任我們的情況下驗證。

### 步驟 1.1：產生簽名金鑰

在生產環境中，代理的簽名金鑰會存放在硬體安全模組 (HSM)、Azure Key Vault 或類似的受保護儲存中。本課程則在記憶體中產生一把新金鑰。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: 建立收據負載

負載包含我們希望收據見證的所有內容：誰執行了操作，使用了什麼工具，帶了什麼參數，回傳了什麼，根據什麼政策，以及何時執行。我們對參數和結果進行雜湊，而不是直接包含它們，以避免收據洩漏敏感內容。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: 簽署並組裝收據

三個步驟：

1. 使用 JCS 對有效負載進行正規化，以確保兩個實現產生相同邏輯收據時，產生的位元組完全一致。
2. 使用 SHA-256 對正規化後的位元組進行雜湊。
3. 使用 Ed25519 私鑰對雜湊進行簽名。

然後將簽名附加到原始有效負載上，產生最終收據。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: 驗證收據

驗證是過程的逆向。我們剝離簽名，重新計算標準雜湊，並利用收據中的公鑰檢查簽名。

執行此驗證的審核員只需要收據本身。無需呼叫任何服務、查詢任何密鑰目錄，也不需要任何信任。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

你應該會看到 `Receipt is valid: True`。代理人已產生其第一個以密碼學簽署的審計紀錄。


## Section 2: 篡改收據

收據的整個目的就是要能夠顯示出是否被篡改過。讓我們來證明這點。

我們將修改收據中的一個字元，然後觀察驗證失敗。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 剛剛發生了什麼？

當我們更改 `policy_id` 時，正規化的位元組也改變了。這些位元組的 SHA-256 雜湊值也隨之改變。簽名（是基於原始雜湊值計算的）不再與新的雜湊值相符。驗證正確地回傳了 `False`。

除非攻擊者擁有私鑰，否則無法修改收據的任何欄位而仍然通過驗證。只要私鑰保存在金鑰庫中，且公開了公鑰，竄改就無法隱藏。

自己試試看：修改上面儲存格中的 `tool_name`、`agent_id` 或 `timestamp`，然後重新執行。每次更改都會導致收據無效。


## Section 3: 將收據串接起來

單一收據保護一個行動。大多數代理會採取多個行動。為了使整個序列能夠顯示被篡改的痕跡，我們在新的收據內文中包含前一張收據的雜湊，將每張收據連結到前一張。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

如果有人移除或重新排序收據，鏈條就會在那正確的點中斷。任何後續收據的驗證都會失敗，因為其 `previous_receipt_hash` 不再與前一個真實雜湊相符。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

現在通過篡改中間的收據來斷開鏈條，然後重新驗證。被篡改的收據未通過其簽名檢查，且下一張收據未通過其鏈接檢查（因為它的 `previous_receipt_hash` 不再匹配被修改的中間收據的哈希值）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

收據 0 仍然通過驗證（它未被修改，且沒有前一筆收據可依賴）。收據 1 因為我們更改了 `tool_args_hash`，導致簽名檢查失敗。收據 2 因其 `previous_receipt_hash` 是根據原始（現已被修改的）收據 1 計算，導致鏈結檢查失敗。

即使攻擊者重新簽署修改過的收據 1（他們無法在沒有私鑰的情況下做到），收據 2 中的鏈結不匹配仍會暴露竄改行為。要隱藏變更，攻擊者必須從修改點開始重新簽署每張收據，這需要持有私鑰。


## Section 4: 用收據簽署包裝代理工具呼叫

在實際部署中，您不希望每位代理編寫者都記得要呼叫 `make_receipt`。您希望每次工具調用都能自動進行收據簽署。

這裡是最簡單的模式：一個包裝類別，接受任何可呼叫的工具函數，並回傳會發出收據的版本。這可適用於任何代理框架，包括 Microsoft Agent Framework（`agent_framework.azure`）。

如果您尚未設置 Azure AI Foundry 專案，下面的本地模擬仍然展示了這個模式。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### 與 Microsoft Agent Framework 整合

上面的 `ReceiptedTool` 包裝器與框架無關。要在使用 Microsoft Agent Framework 建立的代理中使用它，請將包裝後的函式註冊為工具。以下是示意（您會將模擬替換為真實的 Azure AI Foundry 工具註冊）：

```python
# 顯示整合形態的偽代碼。
# 從 agent_framework.azure 匯入 AzureAIProjectAgentProvider
# 從 azure.identity 匯入 AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="你是 Contoso 旅遊代理 ..."
#     tools=[receipted_lookup],   # 被包裝的工具，不是原始函式
# )
# response = agent.run("查詢六月份從雪梨飛往洛杉磯的航班。")
#
# # 執行後，代理所呼叫的每個工具都有簽署的收據：
# audit_chain = receipted_lookup.receipts
```

代理框架不需要了解任何有關收據的資訊。收據簽署是包裝在工具周圍的，而不是內嵌於框架中。這就是您如何在不重寫代理的情況下，為現有代理代碼新增來源訊息。


## 回顧與進階挑戰

您已經：

- 生成了一組 Ed25519 金鑰對。
- 建立並簽署代理工具呼叫的收據。
- 僅使用公鑰離線驗證收據。
- 篡改收據並觀察驗證失敗。
- 建立三份收據的雜湊鏈序列。
- 篡改鏈中間，並觀察簽名失敗與鏈結失敗。
- 將工具函式包裝為自動簽署收據。

**進階挑戰。** 擴充收據結構，新增 `request_id` 欄位（用於分散式追蹤的 UUID）。更新 `make_receipt` 包含該欄位，並確認收據仍能端對端驗證。然後在簽署後修改該欄位，並確認驗證失敗。這會讓你內化每個位元組的規範編碼如何影響簽名。

**重要界限。** 收據證明三件事且只有三件事：歸屬（這把金鑰簽署了此內容）、完整性（內容自簽署後未更改）、以及排序（此收據在另一收據之後）。它們並不證明代理的行為正確、`policy_id` 指定的政策確實被評估，或代理遵守了所有規則。收據是基礎。治理是你建立在上方的系統。

請帶著這個界限重新閱讀課程 README。團隊對收據最常見的誤解是以為「有收據」就代表「已治理」，事實並非如此。收據讓代理行為可審計，卻不能保證其正確。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
